In [505]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

In [506]:
def sigmoid(x):
        return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
        return sigmoid(x) * (1 - sigmoid(x))

In [507]:
def network_initialize(input_feauter, n_hiddens=[3], output_feature=1):
    network = []
    network.append({'weights': np.random.uniform(-1, 1, size=(input_feauter+1, n_hiddens[0]))})
    for i in range(1, len(n_hiddens)):
        network.append({'weights': np.random.uniform(0, 1, size=(n_hiddens[i-1]+1, n_hiddens[i]))})
    network.append({'weights': np.random.uniform(0, 1, size=(n_hiddens[-1]+1, output_feature))})
    return network


# network_initialize(3, [2,2], 1)

In [508]:
def feedforward(network, X):
    net = np.dot(X, network[0]['weights'][:-1]) + network[0]['weights'][-1]
    network[0]['net'] = net
    network[0]['out'] = sigmoid(net)
    for i in range(1, len(network)):
        net = np.dot(network[i - 1]['out'], network[i]['weights'][:-1]) + network[i]['weights'][-1]
        network[i]['net'] = net
        network[i]['out'] = sigmoid(net)
    return network


# network = network_initialize(1, [2], 1)
# print(network)
# network = feedforward(network, [[2],[3]])
# print(network)

In [509]:

def backpropagation(network, X, y, learning_rate):
    # Forward pass
    network = feedforward(network, X)
    
    # Calculate error for the output layer
    error = -2*(y - network[-1]['out'])
    # error = - (y - network[-1]['out']) / (network[-1]['out'] * (1 - network[-1]['out']))
    delta_output = error * sigmoid_derivative(network[-1]['out'])
    # Update weights for the output layer

    network[-1]['weights'][:-1] -= learning_rate * np.dot(network[-2]['out'].T, delta_output)
    network[-1]['weights'][-1]  -= learning_rate * np.sum(delta_output, axis=0)  # Update bias
    
    # Calculate error for hidden layers and update weights
    for i in range(len(network) - 2, 0, -1):
        error = np.dot(delta_output, network[i + 1]['weights'][:-1].T)
        delta_hidden = error * sigmoid_derivative(network[i]['out'])
        network[i]['weights'][:-1] -= learning_rate * np.dot(network[i - 1]['out'].T, delta_hidden)
        network[i]['weights'][-1]  -= learning_rate * np.sum(delta_hidden, axis=0)  # Update bias
        delta_output = delta_hidden
    
    # Update weights for the first hidden layer
    delta_output = delta_output
    error = np.dot(delta_output, network[1]['weights'][:-1].T)
    delta_hidden = error * sigmoid_derivative(network[0]['out'])
    network[0]['weights'][:-1] -= learning_rate * np.dot(X.T, delta_hidden)
    network[0]['weights'][-1]  -= learning_rate * np.sum(delta_hidden, axis=0)  # Update bias
    
    return network

In [510]:
def train(X, y, epochs, learning_rate, test_x, test_y):
    network = network_initialize(input_feauter=X.shape[1], n_hiddens=[3, 3, 3], output_feature=1)
    for epoch in range(epochs):
        network = backpropagation(network, X, y, learning_rate)
        # print(network)
        if epoch % 50 == 0:
            print(f"epoch {epoch} finished", end=' | test_acc:')
            y_pred = prediction(test_x, network)
            acc = accuracy_score(test_y, y_pred)
            print(acc, end=' | train_acc:')
            y_pred2 = prediction(X, network)
            acc = accuracy_score(y, y_pred2)
            print(acc, end=' | loss:')
            print(np.mean(cross_entropy_loss(test_y, y_pred)))
            if acc == 1: break
            
    return network

def prediction(X, trained_network):
    y_pred = np.round(feedforward(trained_network, X)[-1]['out'])
    return y_pred

def cross_entropy_loss(y_true, y_pred):
    # Clip y_pred to prevent log(0) errors
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    # Compute cross-entropy
    loss = - (y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss
        
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y = np.array([[0], [1], [1], [0]])
# trained_network = train(X, y, 10000, .8, X, y)

In [511]:
def load_data(): 
    train_loaded = np.load(f'train_data.npz')
    test_loaded = np.load(f'test_data.npz')
    
    train_x, train_y = train_loaded['x'].T, train_loaded['y'] 
    test_x, test_y = test_loaded['x'].T, test_loaded['y'] 

    # Standarize data
    train_x = train_x / 255.
    test_x = test_x / 255.

    # Shuffle data (no effect for full batch method)
    train_indices = np.random.permutation(train_x.shape[1])
    train_x = train_x[:, train_indices]
    train_y = train_y[train_indices]

    test_indices = np.random.permutation(test_x.shape[1])
    test_x = test_x[:, test_indices]
    test_y = test_y[test_indices]

    return train_x.T, train_y.reshape(-1, 1), test_x.T, test_y.reshape(-1, 1)

train_x, train_y, test_x, test_y = load_data()
trained_network = train(train_x, train_y, 1000, 0.001, test_x, test_y)

epoch 0 finished | test_acc:0.5 | train_acc:0.5 | loss:17.26978799617044
epoch 50 finished | test_acc:0.5175 | train_acc:0.513 | loss:16.664963608531558
epoch 100 finished | test_acc:0.52 | train_acc:0.515 | loss:16.578614668550703
epoch 150 finished | test_acc:0.52 | train_acc:0.5185 | loss:16.578614668550703
epoch 200 finished | test_acc:0.525 | train_acc:0.519 | loss:16.405918787582575
epoch 250 finished | test_acc:0.54 | train_acc:0.5285 | loss:15.887837141658915
epoch 300 finished | test_acc:0.56 | train_acc:0.547 | loss:15.197061613760702
epoch 350 finished | test_acc:0.6275 | train_acc:0.616 | loss:12.86569420710423
epoch 400 finished | test_acc:0.7825 | train_acc:0.756 | loss:7.512357778334142
epoch 450 finished | test_acc:0.7275 | train_acc:0.698 | loss:9.41203445791289
epoch 500 finished | test_acc:0.695 | train_acc:0.681 | loss:10.53457067766397
epoch 550 finished | test_acc:0.7025 | train_acc:0.6815 | loss:10.27552385772141
epoch 600 finished | test_acc:0.7425 | train_acc:0